# Lab 02: Multi-Step Workflow

**Goal:** Build a workflow with multiple processing steps to see how state flows through a chain of nodes.

**What you'll learn:**
- How state accumulates as it passes through nodes
- How each node can read fields set by previous nodes
- Building a realistic multi-step data processing pipeline
- Inspecting state at each stage

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

## Step 1: Define the state for a support ticket pipeline

In [ ]:
class TicketState(TypedDict):
    raw_input: str
    cleaned_input: str
    category: str
    priority: str
    response: str

## Step 2: Create processing nodes

Each node handles one step of the pipeline.
They read from state and return updates.

In [ ]:
def clean_input(state: TicketState) -> dict:
    """Remove extra whitespace and lowercase the input."""
    cleaned = " ".join(state["raw_input"].split()).lower().strip()
    print(f"  [clean_input] '{state['raw_input']}' \u2192 '{cleaned}'")
    return {"cleaned_input": cleaned}

def classify(state: TicketState) -> dict:
    """Classify the ticket into a category."""
    text = state["cleaned_input"]
    if any(w in text for w in ["leave", "sick", "wfh", "vacation"]):
        category = "hr"
    elif any(w in text for w in ["deploy", "bug", "server", "database", "code"]):
        category = "tech"
    elif any(w in text for w in ["expense", "reimburse", "invoice", "bill"]):
        category = "finance"
    else:
        category = "general"
    print(f"  [classify] '{text}' \u2192 category: {category}")
    return {"category": category}

def assign_priority(state: TicketState) -> dict:
    """Assign priority based on keywords."""
    text = state["cleaned_input"]
    if any(w in text for w in ["urgent", "down", "critical", "blocked"]):
        priority = "HIGH"
    elif any(w in text for w in ["help", "issue", "problem", "error"]):
        priority = "MEDIUM"
    else:
        priority = "LOW"
    print(f"  [assign_priority] \u2192 {priority}")
    return {"priority": priority}

def generate_response(state: TicketState) -> dict:
    """Generate a response based on category and priority."""
    responses = {
        "hr": "Your HR request has been forwarded to the HR team.",
        "tech": "A tech support ticket has been created.",
        "finance": "Your finance query has been sent to the accounts team.",
        "general": "Your request has been logged. We'll get back to you shortly.",
    }
    base = responses.get(state["category"], responses["general"])
    response = f"[{state['priority']}] {base} (Category: {state['category']})"
    print(f"  [generate_response] \u2192 {response}")
    return {"response": response}

## Step 3: Build the pipeline graph

In [ ]:
graph = StateGraph(TicketState)
graph.add_node("clean", clean_input)
graph.add_node("classify", classify)
graph.add_node("prioritize", assign_priority)
graph.add_node("respond", generate_response)

graph.add_edge(START, "clean")
graph.add_edge("clean", "classify")
graph.add_edge("classify", "prioritize")
graph.add_edge("prioritize", "respond")
graph.add_edge("respond", END)

app = graph.compile()
print("Graph: START \u2192 clean \u2192 classify \u2192 prioritize \u2192 respond \u2192 END")

## Step 4: Run with test tickets

In [ ]:
test_tickets = [
    "I need to apply for   sick LEAVE   next week",
    "URGENT: production server is DOWN!",
    "How do I submit my   expense   report?",
    "Where is the office cafeteria?",
]

for ticket in test_tickets:
    print(f"\nTicket: '{ticket}'")
    result = app.invoke({"raw_input": ticket})
    print(f"  Final state:")
    print(f"    category: {result['category']}")
    print(f"    priority: {result['priority']}")
    print(f"    response: {result['response']}")

## Step 5: Inspect the full state

In [ ]:
result = app.invoke({"raw_input": "Urgent help needed with database deployment"})
print("Complete state after all nodes:")
for key, value in result.items():
    print(f"  {key}: {value}")

## TODO 1: Add a validation node

Add a node called `"validate"` that runs BEFORE `"classify"`.
It should check if the `cleaned_input` is at least 5 characters.
If too short, set the response to `"Input too short"` and
category to `"invalid"`.

Graph: `START \u2192 clean \u2192 validate \u2192 classify \u2192 prioritize \u2192 respond \u2192 END`

In [ ]:
# def validate(state):
#     if len(state["cleaned_input"]) < 5:
#         return {"category": "___", "response": "___"}
#     return {}
#
# Test with: app.invoke({"raw_input": "Hi"})

## TODO 2: Add a logging node at the end

Add a `"log"` node that runs after `"respond"` (before END).
It should create a log entry string combining all fields:
`"[TIMESTAMP] [PRIORITY] [CATEGORY] response"`

Add a new state field `"log_entry"` (str) to store it.

In [ ]:
# from datetime import datetime
# def log_ticket(state):
#     entry = f"[{datetime.now().isoformat()}] [{state['priority']}] [{state['category']}] {state['response']}"
#     print(f"  [log] {entry}")
#     return {"___": entry}

## Key Takeaways

- State accumulates as it passes through nodes
- Each node can read fields set by earlier nodes
- Nodes only update the fields they return
- Multi-step pipelines = clean \u2192 classify \u2192 prioritize \u2192 respond
- Print statements inside nodes help debug the flow